# Vehicle Diagnostics - Random Forest Classifier Training & Evaluation


### **Imports & Configuration**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, log_loss
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
INPUT_FILE = 'vehicle_data.csv'
MODEL_FILE = 'vehicle_model.pkl'
CLASS_NAMES = ['Healthy', 'Warning', 'Critical']

### **Load and Prepare Data**

In [ ]:
# Load dataset
if not os.path.exists(INPUT_FILE):
    print(f"Error: {INPUT_FILE} NOT found. Please check your workspace.")
else:
    df = pd.read_csv(INPUT_FILE)
    
    # Preprocessing
    X = df.drop('health_status', axis=1)
    y = df['health_status']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Data loaded! Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

### **Build & Train Random Forest Pipeline**

In [ ]:
print("Training Random Forest Classifier Pipeline...")

# Preprocessing definition
categorical_features = ['component']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numeric_features = ['current_km', 'current_months', 'condition_metric_value']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numeric_features)
    ])
    
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_leaf=10, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Model training complete!")

### **Accuracy & Classification Metrics**

In [4]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

Test accuracy: 97.70%

Classification Report:
              precision    recall  f1-score   support

     Healthy     0.9911    0.9848    0.9879       789
     Warning     0.9973    0.9256    0.9601       403
    Critical     0.9549    0.9950    0.9745       808

    accuracy                         0.9770      2000
   macro avg     0.9811    0.9685    0.9742      2000
weighted avg     0.9777    0.9770    0.9769      2000


#### **Classification Metrics Summary (97.70% Accuracy)**

| Class | Precision | Recall | F1-Score | Support |
| --- | ---: | ---: | ---: | ---: |
| **Healthy** | 0.9911 (99.11%) | 0.9848 (98.48%) | 0.9879 (98.79%) | 789 |
| **Warning** | 0.9973 (99.73%) | 0.9256 (92.56%) | 0.9601 (96.01%) | 403 |
| **Critical** | 0.9549 (95.49%) | 0.9950 (99.50%) | 0.9745 (97.45%) | 808 |
| **Macro Avg** | 0.9811 (98.11%) | 0.9685 (96.85%) | 0.9742 (97.42%) | 2000 |
| **Weighted Avg** | 0.9777 (97.77%) | 0.9770 (97.70%) | 0.9769 (97.69%) | 2000 |

### **Confusion Matrix Visualization**

In [ ]:
cm = confusion_matrix(
    y_test,
    model.predict(X_test),
    labels=[0, 1, 2]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Healthy", "Warning", "Critical"]
)

display.plot(cmap="coolwarm", values_format="d")
plt.title("Confusion Matrix - Random Forest Classifier")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

#### **Confusion Matrix Graph**
![Confusion Matrix](confusion_matrix.png)

### **Learning Curve (Random Forest Classifier)**
Shows model performance (training accuracy vs 5-fold cross-validation accuracy) across increasing training sample sizes.

In [ ]:
print("Generating Learning Curve...")
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='#D9381E', label='Training Score (Accuracy)', linewidth=2)
plt.plot(train_sizes, test_mean, 's-', color='#008080', label='Cross-Validation Score (Accuracy)', linewidth=2)

plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#D9381E')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='#008080')

plt.title('Learning Curve (Random Forest Classifier)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Training Set Size (Number of Samples)', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=300)
plt.show()

#### **Learning Curve Graph**
![Learning Curve](learning_curve.png)

### **Model Loss vs Number of Trees**
Evaluates log-loss reduction on both training and test sets as the number of decision trees (`n_estimators`) increases from 5 to 150.

In [ ]:
print("Generating Model Loss vs Number of Trees graph...")
n_estimators_range = list(range(5, 155, 5))
train_losses = []
test_losses = []

for n_est in n_estimators_range:
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=n_est, max_depth=7, min_samples_leaf=10, random_state=42, n_jobs=-1))
    ])
    clf.fit(X_train, y_train)
    
    train_probs = clf.predict_proba(X_train)
    test_probs = clf.predict_proba(X_test)
    
    train_losses.append(log_loss(y_train, train_probs))
    test_losses.append(log_loss(y_test, test_probs))

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, train_losses, 'o-', color='#D9381E', label='Training Loss (Log Loss)', linewidth=2)
plt.plot(n_estimators_range, test_losses, 's-', color='#008080', label='Test Loss (Log Loss)', linewidth=2)

plt.title('Model Loss vs Number of Trees (Random Forest)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Number of Trees (n_estimators)', fontsize=12)
plt.ylabel('Log Loss (Cross-Entropy Loss)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.savefig('loss_vs_trees.png', dpi=300)
plt.show()

#### **Model Loss vs Number of Trees Graph**
![Model Loss vs Number of Trees](loss_vs_trees.png)

### **Save Model**

In [ ]:
joblib.dump(model, MODEL_FILE)
print(f"Model saved successfully as '{MODEL_FILE}'.")